In [ ]:
# =============================================================================
# Created By:     Kai Metzger
# Created School: Franz-Oberthuer-Schule Wuerzburg
# Created Email:  metzgerkai@franz-oberthuer-schule.de
# Created Date:   Tue Nov 17 20:54 UTC 2026
# Version:        1.1.0
# =============================================================================
"""The Module has been build for training the symbols dataset with images + 
ground truth on a Raspberry Pi 5 with a standard USB camera. An image with a 
resolution of 0.3MPixel to 2MP can be recorded with the Python script 
create-symbols-dataset.py
Files:
- 00 data-info.ipynb shows the data count per dataset.
- 01 create-symbols-dataset.py (camera needed)
- 02 training scripts
- 03 detect-symbols.ipynb

"""

# =============================================================================
# Import
# =============================================================================
import cv2
import os
import numpy as np
from keras import layers, models, utils, callbacks
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Enable Nvidia GPUs by un-commenting this line 
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
XLA_FLAGS="--xla_gpu_cuda_data_dir=/usr/"

# To get reproducable results with the same training setting random seed
SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()

DEBUG = True

In [ ]:
# =============================================================================
# Declare variables / dataset
# =============================================================================
# Change the following paths to your dataset path
home_dir = os.path.expanduser("~")

dataset = ""                                            # CHANGE DATASET NUMBER

dataset_for_training = "dataset" + str(dataset)            
img_dir = home_dir + "/ml_project/datasets/symbols/" + dataset_for_training + "/data"
gt_dir = home_dir + "/ml_project/datasets/symbols/" + dataset_for_training + "/gt"
checkpoint_filepath = home_dir + "/ml_project/datasets/symbols/" + dataset_for_training + "/chpt/"
print(img_dir)

In [ ]:
# =============================================================================
# Dataset related parameters
# =============================================================================
NUM_CLASSES = 4
IMG_HEIGHT = 160
IMG_WIDTH = 160
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
IMG_CHANNELS = 3                                       # 1=:grayscale, 3=:color
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

# =============================================================================
# Hyperparameters
# =============================================================================
BATCH_SIZE = 16
EPOCHS = 300

In [ ]:
# =============================================================================
# Load dataset
# =============================================================================
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_for_training,
    labels="inferred",
    label_mode="categorical",                               # 4 Klassen → 1-hot
    shuffle=True,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_for_training,
    labels="inferred",
    label_mode="categorical",
    shuffle=False,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

In [ ]:
# =============================================================================
# Class names
# =============================================================================
class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)
print("Klassen:", class_names, "\n")

In [ ]:
# =============================================================================
# Preview 9 training images
# =============================================================================
plt.figure(figsize=(8, 8))
for images, labels in train_dataset.take(1):
    for i in range( min(BATCH_SIZE, 8)): # choose minimum of batch size and a nuber
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))        
        label_index = labels[i].numpy().argmax()
        plt.title(class_names[label_index])
        plt.axis("off")

plt.tight_layout()

In [ ]:
# =============================================================================
# Split validation dataset into validation and test datasets
# =============================================================================
val_batches = tf.data.experimental.cardinality(validation_dataset)
test_dataset = validation_dataset.take(val_batches // 4) # 25% of validation dataset for testing
validation_dataset = validation_dataset.skip(val_batches // 4)

In [ ]:
print('Number of training batches: %d' % tf.data.experimental.cardinality(train_dataset))
print('Number of validation batches: %d' % tf.data.experimental.cardinality(validation_dataset))

In [ ]:
# =============================================================================
# Prefetch datasets for performance
# =============================================================================
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

print('Number of testing batches: %d' % tf.data.experimental.cardinality(test_dataset))

In [ ]:
# =============================================================================
# Data Augmentation
# =============================================================================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=0.1),
    layers.RandomZoom(0.05),   
])

In [ ]:
for image, _ in train_dataset.take(1):
  # Display the original image and 9 augmented versions
  plt.figure(figsize=(8, 8))
  first_image = image[0]
  for i in range( min(BATCH_SIZE, 8)):
    ax = plt.subplot(3, 3, i + 1)
    augmented_image = data_augmentation(tf.expand_dims(first_image, 0))
    plt.imshow(augmented_image[0] / 255)
    plt.axis('off')

plt.tight_layout()

In [ ]:
# =============================================================================
# Create neural network with 1, 2, ..., L layers 
# and (64, 32, 16, 4) filters per layer.
# =============================================================================
model = models.Sequential()

model_name = "small"
if model_name == "small":
    model = models.Sequential([
        # INPUT LAYER
        layers.Input(shape=INPUT_SHAPE),

        # Add rescaling here if pixel values are in [0, 255].
        layers.Rescaling(1./255),
        
        # For RGB input, you can optionally convert to grayscale:
        # layers.Lambda(tf.image.rgb_to_grayscale),

        # DATA AUGMENTATION
        data_augmentation,

        # HIDDEN LAYER 1: 32 filters
        layers.Conv2D(filters=16, kernel_size=3, padding='same', 
                      activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),

        # HIDDEN LAYER 2: 128 filters
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(), 

        # HIDDEN LAYER 2: 128 filters
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),          
        
        # HIDDEN LAYER 2: 128 filters
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),  

        # HIDDEN LAYER 3: 16 filters
        layers.Conv2D(16, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),

        # Vectorize the 2D feature maps into a 1D feature vector
        layers.Flatten(),

        # HIDDEN LAYER 4: Fully connected layer with 256 neurons
        layers.Dense(256,activation='relu'),
        layers.Dropout(0.3),        
        
        layers.Dense(32,activation='relu'),
        layers.Dropout(0.3),

        # OUTPUT LAYER
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ])
elif model_name ==  "lenet":
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),
        data_augmentation,

        # Add Rescaling(1./255) here if pixel values are in [0, 255].
        # For RGB input, you can optionally convert to grayscale:
        # layers.Lambda(tf.image.rgb_to_grayscale),

        layers.Conv2D(6, kernel_size=5, activation="tanh", padding="valid"),
        layers.AveragePooling2D(pool_size=2, strides=2),

        layers.Conv2D(16, kernel_size=5, activation="tanh", padding="valid"),
        layers.AveragePooling2D(pool_size=2, strides=2),

        layers.Flatten(),
        layers.Dense(120, activation="tanh"),
        layers.Dense(84, activation="tanh"),
        layers.Dense(4, activation="softmax"),
    ])


In [ ]:
# =============================================================================
# HYPERPARAMETERS
# =============================================================================
LEARNING_RATE = 3e-4

lr = "fixed"                                                 # HYPERPARAMETER 4
if lr == "fixed":
    learning_rate = LEARNING_RATE 
elif lr == "decay":
    learning_rate = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LEARNING_RATE,
        decay_steps=500, # 10% every 10 epochs = ???
        decay_rate=0.9
    )  
elif lr == "plateu":                                    
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
    )


optimizer = "adam"                                           # HYPERPARAMETER 5
if optimizer == "adam":
    optimizer_fcn = keras.optimizers.Adam(learning_rate=learning_rate)
elif optimizer == "sgd":
    optimizer_fcn = keras.optimizers.SGD(learning_rate=learning_rate)
elif optimizer == "rmsprop":
    optimizer_fcn = keras.optimizers.RMSprop(learning_rate=learning_rate)
elif optimizer == "adagrad":
    optimizer_fcn = keras.optimizers.Adagrad(learning_rate=learning_rate)

loss_fcn = "categorical_crossentropy"       
#loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)                    
#loss = "sparse_categorical_crossentropy"
#loss = "mean_squared_error"
#...

metric_fcn = ["accuracy"]
#metric=["sparse_categorical_accuracy"]
#...

model.compile(
    loss=loss_fcn,
    optimizer=optimizer_fcn,
    metrics=metric_fcn,
)

In [ ]:
# =============================================================================
# Take a look at our model
# =============================================================================
if DEBUG:
    print(model.summary())

In [ ]:
# AUTOTUNE
# =============================================================================
# TensorBoard Logging
# =============================================================================
log_path = "logs/fit"
tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=log_path,
    histogram_freq=1
)

# Starten in Shell: tensorboard --logdir datasets/symbols/logs/fit

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
)

In [ ]:
# =============================================================================
# Callback for training
# =============================================================================
# Set verbose flag to 0 to omit loss/accuracy output per epoch
checkpoint = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath + "chpt.keras",
    monitor='val_loss',       # metric to monitor
    save_best_only=True,      # only save when metric improves
    mode='auto',              # minimize loss and aximize accuracy
    verbose=0,                # Print a line when the model is saved
)

In [ ]:
# =============================================================================
# Train the neuronal network e.g. for some epochs.
# =============================================================================
history = model.fit(
    train_dataset, 
    validation_data=validation_dataset,
    epochs=EPOCHS, 
    verbose=1, 
    shuffle=False,
    callbacks=[checkpoint, 
               early_stopping,
               tensorboard_callback,
               ])

In [ ]:
# =============================================================================
# Plot train and val accuracy.
# =============================================================================

plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.show()


In [ ]:
# =============================================================================
# Plot train and val loss.
# =============================================================================

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='lower left')
plt.show()

In [ ]:
# =============================================================================
# Load pretrained dataset weights to e.g. test on new (unseen) data.
# =============================================================================
model.load_weights(checkpoint_filepath + "chpt.keras")

In [ ]:
# =============================================================================
# Test Accuracy on test set
# =============================================================================
results = model.evaluate(test_dataset, 
                         verbose=1, 
                         return_dict=True,
                         )

loss = results["loss"]
accuracy = results.get("accuracy")  # available if you compiled with an accuracy metric

print("Test loss:", loss)
print("Test accuracy:", accuracy)

In [ ]:
# =============================================================================
# Konfusionsmatrix + Klassifizierungsreport
# =============================================================================
# Daten aus Validation/Test holen
y_true = []
y_pred = []

for images, labels in validation_dataset:               # Get predictions & gt
    preds = model.predict(images)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Classes:", class_names)                          # Class list

cm = confusion_matrix(y_true, y_pred)                   # Confusion Matrix

plt.figure(figsize=(8, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# TODO
# =============================================================================
# Testing on a single image.
# =============================================================================
data_pred = np.zeros((1, img_size_x, img_size_y), dtype=float)
img_pred = cv2.imread(home_dir + "/ml_project/datasets/symbols/" + dataset_for_training + "/data/14.png")
img_pred = pic_prep(img_pred, img_size_x, img_size_y)

img_pred = utils.to_categorical(img_pred)

data_pred[0,:,:] = img_pred
data_pred = data_pred.reshape(1,img_dim)
result = model.predict(data_pred)
result = np.round(result, decimals=2)

print("Probability for classes: (cross, circle, triangle, square) in percent", 
      result)

In [ ]:
# TODO
# =============================================================================
# Output class: 
# translate class label (0,1,2,3) to class (cross, circle, triangle, square).
# =============================================================================

max_res = 0
res_index = 4
for i in range(0, 4, 1):
    if result[0,i] > max_res:
        max_res = result[0,i]
        res_index = i

if res_index == 0:
    print('Cross detected!')
elif res_index == 1:
    print('Circle detected!')
elif res_index == 2:
    print('Triangle detected!')
elif res_index == 3:
    print('Square detected!')
elif res_index == 4:
    print('Error!')